# Regex in Data Cleaning

## Overview
Clinical data is notoriously messy. Dosage strings (`'10mg'`, `'10 MG'`, `'2.5 mg/day'`), phone numbers in multiple formats, lab values with qualifiers (`'>60'`, `'<5.7%'`), and free-text notes all require regex to clean at scale.

**The SQL data cleaning workflow:**
```sql
-- 1. Identify bad data
SELECT * FROM medications WHERE dosage !~ '^[0-9]+(\.[0-9]+)?(mg|mcg|g)$';

-- 2. Preview the fix (SELECT first — never UPDATE blind)
SELECT dosage,
       regexp_replace(lower(regexp_replace(dosage,' ','','g')), '/\w+$', '') AS clean
FROM medications;

-- 3. Apply in a transaction
BEGIN;
UPDATE medications
   SET dosage = regexp_replace(lower(regexp_replace(dosage,' ','','g')), '/\w+$', '')
 WHERE dosage IS NOT NULL;

-- 4. Validate
SELECT COUNT(*) FROM medications WHERE dosage !~ '^[0-9]+(\.[0-9]+)?(mg|mcg|g)$';
COMMIT;   -- or ROLLBACK if validation fails
```

---

In [ ]:
import sqlite3, re

def make_conn():
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    conn.create_function("regexp", 2,
        lambda p, s: bool(re.search(p, s or "", re.IGNORECASE)))
    conn.create_function("regexp_replace", 3,
        lambda p, s, r: re.sub(p, r, s or "", flags=re.IGNORECASE))
    return conn

conn = make_conn()
conn.executescript("""
CREATE TABLE medications (
    med_id INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id TEXT NOT NULL, drug_name TEXT NOT NULL,
    dosage TEXT, frequency TEXT
);
CREATE TABLE lab_results (
    lab_id INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id TEXT NOT NULL, test_name TEXT NOT NULL,
    raw_value TEXT NOT NULL, result_date TEXT NOT NULL
);
CREATE TABLE patients (
    patient_id TEXT PRIMARY KEY, full_name TEXT NOT NULL,
    phone TEXT, dob TEXT, mrn TEXT
);
INSERT INTO medications VALUES
    (NULL,'P001','Lisinopril','10mg','once daily'),
    (NULL,'P001','Metformin','500 mg','BID'),
    (NULL,'P002','Fluticasone','250mcg','BD'),
    (NULL,'P003','Amlodipine','5 mg','once daily'),
    (NULL,'P005','Metformin','1g','BD'),
    (NULL,'P005','Insulin glargine','20 units','nocte'),
    (NULL,'P001','Atorvastatin','40MG','once daily'),
    (NULL,'P003','Ramipril','2.5 mg/day','once daily');
INSERT INTO lab_results VALUES
    (NULL,'P001','HbA1c','7.2%','2023-10-01'),
    (NULL,'P001','eGFR','>60 mL/min','2023-10-01'),
    (NULL,'P002','SpO2','94%','2023-05-15'),
    (NULL,'P005','HbA1c','8.9 %','2023-11-10'),
    (NULL,'P005','HbA1c','<5.7%','2022-01-01'),
    (NULL,'P001','BP','142/88 mmHg','2023-04-10'),
    (NULL,'P001','BP','128/82 mmHg','2023-10-01'),
    (NULL,'P004','BP','118/76 mmHg','2023-09-01');
INSERT INTO patients VALUES
    ('P001','Aroha Ngata',      '506-555-0101','1985-03-15','MRN-004521'),
    ('P002','Liam Chen',        '(416) 555-2233','1990-07-22','MRN-008823'),
    ('P003','Fatima Al-Rashid', '604.555.9900','1978-11-05','MRN-012234'),
    ('P004','James MacLeod',    '902 555 7788','2001-01-30','MRN-003344'),
    ('P005','Mei-Ling Tran',    '514-555-1122','1965-08-19','MRN-019900'),
    ('P006','Bad Data',         'not-a-phone','not-a-date',NULL);
""")
conn.commit()
print("Cleaning dataset ready")
for t in ("medications","lab_results","patients"):
    print(f"  {t}: {conn.execute('SELECT COUNT(*) FROM '+t).fetchone()[0]} rows")


---
## Dosage normalisation

In [ ]:
import re

print("=== Dosage normalisation: raw -> clean ===")
rows = conn.execute("SELECT drug_name, dosage FROM medications ORDER BY drug_name").fetchall()

def normalise_dosage(raw):
    if not raw:
        return None
    s = raw.strip()
    m = re.match(r'^([0-9]+(?:[.][0-9]+)?)\s*(mg|mcg|g|units|MG|MCG)(/\w+)?$', s, re.IGNORECASE)
    if m:
        return m.group(1) + m.group(2).lower() + (m.group(3) or '')
    return s

print(f"  {'Drug':<22s}  {'Raw':<14s}  Normalised")
print("  " + "-"*55)
for r in rows:
    raw  = r['dosage'] or ''
    norm = normalise_dosage(raw)
    arrow = ' ->' if raw != norm else ' = '
    print(f"  {r['drug_name']:<22s}  {raw!r:<14s}{arrow}  {norm!r}")

print()
print("PostgreSQL equivalent:")
print("  UPDATE medications")
print("     SET dosage = regexp_replace(lower(regexp_replace(dosage,' ','','g')),'/\\w+$','')")
print("   WHERE dosage IS NOT NULL;")


---
## Lab result parsing

In [ ]:
import re

print("=== Lab result parsing: extract qualifier, value, unit ===")
rows = conn.execute(
    "SELECT patient_id, test_name, raw_value FROM lab_results ORDER BY lab_id"
).fetchall()

def parse_lab(raw):
    raw = (raw or '').strip()
    m = re.match(r'^([<>~]?)\s*([0-9]+(?:[.][0-9]+)?)\s*([%a-zA-Z/0-9.]*)', raw)
    if m:
        return (m.group(1) or '='), float(m.group(2)), m.group(3).strip()
    return None, None, None

print(f"  {'test':<10s}  {'raw':<18s}  {'qualifier':<10s}  {'value':<8s}  unit")
print("  " + "-"*62)
for r in rows:
    q, v, u = parse_lab(r['raw_value'])
    print(f"  {r['test_name']:<10s}  {r['raw_value']!r:<18s}  {str(q):<10s}  {str(v):<8s}  {u!r}")

print()
print("PostgreSQL (extract numeric part):")
print("  SELECT test_name, raw_value,")
print("         (regexp_match(raw_value, '[0-9]+(?:[.][0-9]+)?'))[1]::numeric AS num_val,")
print("         regexp_replace(raw_value, '[<>~]?\\s*[0-9]+(?:[.][0-9]+)?\\s*', '') AS unit")
print("  FROM lab_results;")


---
## Phone normalisation

In [ ]:
import re

print("=== Phone number normalisation to NNN-NNN-NNNN ===")
rows = conn.execute(
    "SELECT patient_id, phone FROM patients ORDER BY patient_id"
).fetchall()

def normalise_phone(raw):
    if not raw:
        return None
    digits = re.sub(r'[^0-9]', '', raw)
    if len(digits) == 10:
        return f"{digits[0:3]}-{digits[3:6]}-{digits[6:10]}"
    return None

print(f"  {'patient_id':<12s}  {'raw':<22s}  {'normalised':<18s}  status")
print("  " + "-"*65)
for r in rows:
    raw    = r['phone'] or ''
    norm   = normalise_phone(raw)
    status = 'valid' if norm else 'INVALID'
    print(f"  {r['patient_id']:<12s}  {raw!r:<22s}  {str(norm)!r:<18s}  {status}")

print()
print("PostgreSQL:")
print("  SELECT patient_id,")
print("    CASE WHEN length(regexp_replace(phone,'[^0-9]','','g')) = 10")
print("      THEN regexp_replace(regexp_replace(phone,'[^0-9]','','g'),")
print("           '([0-9]{3})([0-9]{3})([0-9]{4})','\\1-\\2-\\3')")
print("      ELSE NULL END AS normalised_phone")
print("  FROM patients;")


---
## Common Pitfalls

**1. Cleaning row-by-row in Python instead of bulk SQL UPDATE**
Fetching all rows, cleaning in Python, and writing back creates N round-trips for N rows. A single `UPDATE medications SET dosage = regexp_replace(...)` processes all rows in one pass.

**2. regexp_replace on NULL returning NULL**
In PostgreSQL, `regexp_replace(NULL, ' ', '')` returns NULL. Use `COALESCE`: `regexp_replace(COALESCE(dosage,''), ' ', '')`.

**3. Casting lab values without stripping qualifier prefix**
`'<5.7%'::numeric` raises `invalid input syntax`. Strip first: `(regexp_match(raw_value,'[0-9]+(?:[.][0-9]+)?'))[1]::numeric`.

**4. Phone normalisation stripping country code**
`regexp_replace('+1-506-555-0101','[^0-9]','','g')` -> `15065550101` (11 digits). Strip a leading `1` from 11-digit results or handle +1 explicitly.

**5. Running UPDATE without previewing as SELECT first**
Always run the SELECT version of a cleaning query before executing UPDATE. A wrong regex in an UPDATE can corrupt an entire column. Wrap in `BEGIN`/`ROLLBACK` during testing.

**6. Not validating after cleaning**
Always run a post-cleaning validation query to confirm the cleaned values match the expected format. A regex that almost works may silently leave edge cases uncleaned.


---
*sql_methods_library - Samantha McGarrigle*